# Export & Serialization: Save and Load Results

Complete guide to persisting segmentation results:

1. CSV export for sharing and analysis
2. JSON serialization for portability
3. Pickle serialization for full state
4. Loading and validating saved results

## Setup

In [1]:
import sys
from pathlib import Path

# Add src to path for importing pso_segmentation
sys.path.insert(0, str(Path("..") / "src"))

import numpy as np
import pandas as pd

from pso_segmentation import (
    OptimizerConfig,
    SegmentationOptimizer,
    example_fitness_r2_with_all_constraints,
    export_segmentation_to_csv,
    import_segmentation_from_csv,
    load_optimizer_state,
    save_optimizer_state,
)

np.random.seed(42)
print("Libraries imported successfully!")

Libraries imported successfully!


## Generate Sample Data

In [2]:
n_samples = 2000
scores = np.random.beta(a=2, b=5, size=n_samples)
labels = (np.random.rand(n_samples) < scores).astype(int)

print(f"Dataset: {n_samples} samples")
print(f"Default rate: {labels.mean():.1%}")

Dataset: 2000 samples
Default rate: 28.4%


## Run Segmentation

In [3]:
config = OptimizerConfig(n_segments=4, pop_size=80, max_iter=300, seed=42)

optimizer = SegmentationOptimizer(config)
optimizer.fit(
    scores, labels, lambda cuts: example_fitness_r2_with_all_constraints(cuts, scores, labels)
)

result = optimizer.get_metrics()
print(f"Segmentation complete - R²: {result.r2:.4f}")

Segmentation complete - R²: 0.1345


## 1. CSV Export

Export to CSV files for sharing and analysis:

In [4]:
import os

# Create output directory
output_dir = "./segmentation_output"
Path(output_dir).mkdir(exist_ok=True)

print("Exporting to CSV...")
files = export_segmentation_to_csv(
    cuts=optimizer.get_cuts(), scores=scores, labels=labels, output_dir=output_dir
)

print("\nExported files:")
for key, file_path in files.items():
    print(f"  - {key}: {file_path}")
    file_size = os.path.getsize(file_path)
    print(f"    Size: {file_size:,} bytes")

Exporting to CSV...

Exported files:
  - cuts: segmentation_output\cuts.csv
    Size: 90 bytes
  - data: segmentation_output\segmented_data.csv
    Size: 53,114 bytes
  - metrics: segmentation_output\segment_metrics.csv
    Size: 353 bytes


## 2. Inspect CSV Files

In [5]:
# Load and inspect cuts.csv
print("CUTS.CSV:")
print("-" * 50)
df_cuts = pd.read_csv(f"{output_dir}/cuts.csv")
print(df_cuts.to_string())

# Load and inspect segmented_data.csv
print("\n\nSEGMENTED_DATA.CSV (first 10 rows):")
print("-" * 50)
df_segmented = pd.read_csv(f"{output_dir}/segmented_data.csv")
print(df_segmented.head(10).to_string())

print(f"\nShape: {df_segmented.shape}")
print(f"Columns: {list(df_segmented.columns)}")

# Load and inspect segment_metrics.csv
print("\n\nSEGMENT_METRICS.CSV:")
print("-" * 50)
df_metrics = pd.read_csv(f"{output_dir}/segment_metrics.csv")
print(df_metrics.to_string())

CUTS.CSV:
--------------------------------------------------
   cut_index  cut_value
0          0   0.186670
1          1   0.315389
2          2   0.453364


SEGMENTED_DATA.CSV (first 10 rows):
--------------------------------------------------
      score  label  segment
0  0.353677    1.0        2
1  0.248558    0.0        1
2  0.415959    1.0        2
3  0.159968    1.0        0
4  0.550283    0.0        3
5  0.110945    0.0        0
6  0.509897    0.0        3
7  0.177270    0.0        0
8  0.198290    0.0        1
9  0.376237    0.0        2

Shape: (2000, 3)
Columns: ['score', 'label', 'segment']


SEGMENT_METRICS.CSV:
--------------------------------------------------
   segment  n_observations  proportion   pd_rate  min_score  max_score
0        0             597      0.2985  0.120603   0.005368   0.186340
1        1             593      0.2965  0.219224   0.186746   0.314855
2        2             470      0.2350  0.344681   0.315482   0.453227
3        3             340     

## 3. CSV Import

Load segmentation results from CSV:

In [6]:
# Save optimizer to JSON
import json

json_path = f"{output_dir}/optimizer.json"
print(f"Saving to JSON: {json_path}")
optimizer.to_json(json_path)
print("✓ Saved")

# Inspect JSON file
with open(json_path) as f:
    json_data = json.load(f)

print("\nJSON structure:")
for key in json_data:
    if isinstance(json_data[key], (list, dict)):
        print(f"  {key}: {type(json_data[key]).__name__}")
    else:
        print(f"  {key}: {json_data[key]}")

# Export metrics as JSON directly
metrics_json_path = f"{output_dir}/metrics.json"
print(f"\nExporting metrics to JSON: {metrics_json_path}")
metrics_data = {
    "r2": float(result.r2),
    "n_segments": int(result.n_segments),
    "h_inter": float(result.h_inter),
    "h_intra": float(result.h_intra),
    "pd_by_segment": [float(x) for x in result.pd_by_segment],
    "segment_proportions": [float(x) for x in result.segment_proportions],
}
with open(metrics_json_path, "w") as f:
    json.dump(metrics_data, f, indent=2)
print("✓ Saved")

with open(metrics_json_path) as f:
    metrics_json = json.load(f)

print("\nMetrics JSON:")
print(json.dumps(metrics_json, indent=2))

Saving to JSON: ./segmentation_output/optimizer.json
✓ Saved

JSON structure:
  config: dict
  cuts: list
  r2: 0.1344649739545106
  n_segments: 4
  pd_by_segment: list
  segment_sizes: list
  segment_proportions: list
  h_inter: 54.74331296388202
  h_intra: 352.3761870360823

Exporting metrics to JSON: ./segmentation_output/metrics.json
✓ Saved

Metrics JSON:
{
  "r2": 0.1344649739545106,
  "n_segments": 4,
  "h_inter": 54.74331296388202,
  "h_intra": 352.3761870360823,
  "pd_by_segment": [
    0.12060301507535667,
    0.21922428330519067,
    0.34468085106375645,
    0.6029411764704109
  ],
  "segment_proportions": [
    0.29849999999998506,
    0.29649999999998516,
    0.23499999999998825,
    0.1699999999999915
  ]
}


## 4. JSON Export

Export results and optimizer to JSON for portability:

In [11]:
# Save optimizer to JSON
json_path = f"{output_dir}/optimizer.json"
print(f"Saving to JSON: {json_path}")
optimizer.to_json(json_path)
print("✓ Saved")

# Inspect JSON file
with open(json_path) as f:
    json_data = json.load(f)

print("\nJSON structure:")
for key in json_data:
    if isinstance(json_data[key], (list, dict)):
        print(f"  {key}: {type(json_data[key]).__name__}")
    else:
        print(f"  {key}: {json_data[key]}")

# Export metrics as JSON
metrics_json_path = f"{output_dir}/metrics.json"
print(f"\nExporting metrics to JSON: {metrics_json_path}")
metrics_data = {
    "r2": float(result.r2),
    "n_segments": int(result.n_segments),
    "h_inter": float(result.h_inter),
    "h_intra": float(result.h_intra),
    "pd_by_segment": [float(x) for x in result.pd_by_segment],
    "segment_proportions": [float(x) for x in result.segment_proportions],
}
with open(metrics_json_path, "w") as f:
    json.dump(metrics_data, f, indent=2)
print("✓ Saved")

with open(metrics_json_path) as f:
    metrics_json = json.load(f)

print("\nMetrics JSON:")
print(json.dumps(metrics_json, indent=2))

Saving to JSON: ./segmentation_output/optimizer.json
✓ Saved

JSON structure:
  config: dict
  cuts: list
  r2: 0.1344649739545106
  n_segments: 4
  pd_by_segment: list
  segment_sizes: list
  segment_proportions: list
  h_inter: 54.74331296388202
  h_intra: 352.3761870360823

Exporting metrics to JSON: ./segmentation_output/metrics.json
✓ Saved

Metrics JSON:
{
  "r2": 0.1344649739545106,
  "n_segments": 4,
  "h_inter": 54.74331296388202,
  "h_intra": 352.3761870360823,
  "pd_by_segment": [
    0.12060301507535667,
    0.21922428330519067,
    0.34468085106375645,
    0.6029411764704109
  ],
  "segment_proportions": [
    0.29849999999998506,
    0.29649999999998516,
    0.23499999999998825,
    0.1699999999999915
  ]
}


## 5. JSON Import

In [7]:
print("Loading optimizer from JSON...")
optimizer_from_json = SegmentationOptimizer.from_json(json_path)
result_from_json = optimizer_from_json.get_metrics()

print("✓ Loaded")
print("\nComparison:")
print(f"  Original R²: {result.r2:.4f}")
print(f"  Loaded R²: {result_from_json.r2:.4f}")
print(f"  Match: {np.isclose(result.r2, result_from_json.r2)}")

print(f"\nOriginal cuts: {np.round(optimizer.get_cuts(), 3)}")
print(f"Loaded cuts: {np.round(optimizer_from_json.get_cuts(), 3)}")
print(f"Match: {np.allclose(optimizer.get_cuts(), optimizer_from_json.get_cuts())}")

Loading optimizer from JSON...
✓ Loaded

Comparison:
  Original R²: 0.1345
  Loaded R²: 0.1345
  Match: True

Original cuts: [0.187 0.315 0.453]
Loaded cuts: [0.187 0.315 0.453]
Match: True


## 6. Pickle Export

Full state serialization using pickle:

In [8]:
pickle_path = f"{output_dir}/optimizer.pkl"
print(f"Saving to pickle: {pickle_path}")
save_optimizer_state(optimizer, pickle_path)
print("✓ Saved")

file_size = os.path.getsize(pickle_path)
print(f"File size: {file_size:,} bytes")

print("\nComparison of export formats:")
export_formats = {
    "cuts.csv": f"{output_dir}/cuts.csv",
    "metrics.json": f"{output_dir}/metrics.json",
    "optimizer.json": f"{output_dir}/optimizer.json",
    "optimizer.pkl": pickle_path,
}

for name, path in export_formats.items():
    size = os.path.getsize(path)
    print(f"  {name:20s}: {size:>8,} bytes")

Saving to pickle: ./segmentation_output/optimizer.pkl
✓ Saved
File size: 55,380 bytes

Comparison of export formats:
  cuts.csv            :       90 bytes
  metrics.json        :      384 bytes
  optimizer.json      :      825 bytes
  optimizer.pkl       :   55,380 bytes


## 7. Pickle Import

In [9]:
print("Loading optimizer from pickle...")
optimizer_from_pickle = load_optimizer_state(pickle_path)
result_from_pickle = optimizer_from_pickle.get_metrics()

print("✓ Loaded")
print("\nVerification:")
print(f"  Type: {type(optimizer_from_pickle).__name__}")
print(f"  R²: {result_from_pickle.r2:.4f}")
print(f"  Cuts: {np.round(optimizer_from_pickle.get_cuts(), 3)}")

# Verify it works
print("\nSummary from loaded optimizer:")
print(optimizer_from_pickle.summary())

Loading optimizer from pickle...
✓ Loaded

Verification:
  Type: SegmentationOptimizer
  R²: 0.1345
  Cuts: [0.187 0.315 0.453]

Summary from loaded optimizer:
SEGMENTATION OPTIMIZER RESULTS
R² (Variance Explained): 0.1345
Number of Segments: 4

Cut Boundaries:
  Cut 1: 0.186670
  Cut 2: 0.315389
  Cut 3: 0.453364

Segment Statistics:
  Segment 0: PD=12.06%, Size=29.85% (n=597)
  Segment 1: PD=21.92%, Size=29.65% (n=593)
  Segment 2: PD=34.47%, Size=23.50% (n=470)
  Segment 3: PD=60.29%, Size=17.00% (n=340)

Constraint Validation:
  Valid: True


## 8. Export Strategy Comparison

In [10]:
strategies = pd.DataFrame(
    {
        "Format": ["CSV", "JSON", "Pickle"],
        "Use Case": [
            "Sharing, analysis in Excel/Pandas",
            "Web APIs, portability, auditing",
            "Python-only, full state, model reuse",
        ],
        "Pros": [
            "Universal, human-readable",
            "Portable, versioning-friendly",
            "Complete state, fast I/O",
        ],
        "Cons": [
            "Text-based, loses metadata",
            "Slightly larger than pickle",
            "Python-specific, security risk",
        ],
    }
)

print("EXPORT STRATEGY COMPARISON:")
print(strategies.to_string(index=False))

print("\n\nRECOMMENDATIONS:")
print("""
✓ Use CSV when:
  - Sharing results with non-technical users
  - Analyzing in Excel/Tableau
  - Building audit trail
  - Storing in data warehouse

✓ Use JSON when:
  - Integrating with web services
  - Need human-readable config
  - Version control friendly
  - Cross-language support needed

✓ Use Pickle when:
  - Need to reuse optimizer in Python
  - Performance-critical workflows
  - Full model serialization needed
  - Trusted environment only
""")

EXPORT STRATEGY COMPARISON:
Format                             Use Case                          Pros                           Cons
   CSV    Sharing, analysis in Excel/Pandas     Universal, human-readable     Text-based, loses metadata
  JSON      Web APIs, portability, auditing Portable, versioning-friendly    Slightly larger than pickle
Pickle Python-only, full state, model reuse      Complete state, fast I/O Python-specific, security risk


RECOMMENDATIONS:

✓ Use CSV when:
  - Sharing results with non-technical users
  - Analyzing in Excel/Tableau
  - Building audit trail
  - Storing in data warehouse

✓ Use JSON when:
  - Integrating with web services
  - Need human-readable config
  - Version control friendly
  - Cross-language support needed

✓ Use Pickle when:
  - Need to reuse optimizer in Python
  - Performance-critical workflows
  - Full model serialization needed
  - Trusted environment only



## 9. Complete Workflow: Develop → Save → Load

In [14]:
print("COMPLETE WORKFLOW EXAMPLE")
print("=" * 60)

# Phase 1: Development (done above)
print("\n[PHASE 1] Development")
print("-" * 60)
print("✓ Segmentation created")
print(f"  R² = {result.r2:.4f}")
print(f"  Cuts = {np.round(optimizer.get_cuts(), 3)}")

# Phase 2: Export
print("\n[PHASE 2] Export")
print("-" * 60)
export_dir_final = "./production_export"
Path(export_dir_final).mkdir(exist_ok=True)

print("Exporting all formats...")
# CSV
export_segmentation_to_csv(
    cuts=optimizer.get_cuts(), scores=scores, labels=labels, output_dir=export_dir_final
)
print("✓ CSV exported")

# JSON
optimizer.to_json(f"{export_dir_final}/optimizer.json")
metrics_data = {
    "r2": float(result.r2),
    "n_segments": int(result.n_segments),
    "h_inter": float(result.h_inter),
    "h_intra": float(result.h_intra),
    "pd_by_segment": [float(x) for x in result.pd_by_segment],
    "segment_proportions": [float(x) for x in result.segment_proportions],
}
with open(f"{export_dir_final}/metrics.json", "w") as f:
    json.dump(metrics_data, f, indent=2)
print("✓ JSON exported")

# Pickle
save_optimizer_state(optimizer, f"{export_dir_final}/optimizer.pkl")
print("✓ Pickle exported")

# Phase 3: Validate load
print("\n[PHASE 3] Validation")
print("-" * 60)

# Load from different formats
opt_json = SegmentationOptimizer.from_json(f"{export_dir_final}/optimizer.json")
opt_pickle = load_optimizer_state(f"{export_dir_final}/optimizer.pkl")
scores_csv, labels_csv, segs_csv, cuts_csv = import_segmentation_from_csv(
    f"{export_dir_final}/segmented_data.csv", f"{export_dir_final}/cuts.csv"
)

print("✓ Loaded from JSON")
print("✓ Loaded from pickle")
print("✓ Loaded from CSV")

# Verify all match
print("\nVerification:")
print(f"  JSON R² matches: {np.isclose(result.r2, opt_json.get_metrics().r2)}")
print(f"  Pickle R² matches: {np.isclose(result.r2, opt_pickle.get_metrics().r2)}")
print(f"  CSV cuts match: {np.allclose(optimizer.get_cuts(), cuts_csv)}")

print("\n" + "=" * 60)
print("✓ ALL EXPORTS VALIDATED SUCCESSFULLY")

COMPLETE WORKFLOW EXAMPLE

[PHASE 1] Development
------------------------------------------------------------
✓ Segmentation created
  R² = 0.1345
  Cuts = [0.187 0.315 0.453]

[PHASE 2] Export
------------------------------------------------------------
Exporting all formats...
✓ CSV exported
✓ JSON exported
✓ Pickle exported

[PHASE 3] Validation
------------------------------------------------------------
✓ Loaded from JSON
✓ Loaded from pickle
✓ Loaded from CSV

Verification:
  JSON R² matches: True
  Pickle R² matches: True
  CSV cuts match: True

✓ ALL EXPORTS VALIDATED SUCCESSFULLY


## 10. Best Practices

In [13]:
best_practices = """
BEST PRACTICES FOR EXPORT & SERIALIZATION:

1. ORGANIZATION
   ✓ Use consistent directory structure
   ✓ Include metadata (dates, versions)
   ✓ Document format choices
   ✓ Version all artifacts

2. CSV EXPORTS
   ✓ Always export cuts.csv first
   ✓ Then segmented_data.csv for verification
   ✓ Include segment_metrics.csv for documentation
   ✓ Test import immediately after export

3. JSON EXPORTS
   ✓ Use optimizer.to_json() for portability
   ✓ Separately export metrics.json
   ✓ Document schema and versions
   ✓ Validate with JSON schema

4. PICKLE EXPORTS
   ✓ Use only in trusted environments
   ✓ Document Python version used
   ✓ Test loading on target environment
   ✓ Consider security implications

5. FILE MANAGEMENT
   ✓ Use consistent naming: {timestamp}_{modeltype}.pkl
   ✓ Compress older exports
   ✓ Archive with metadata
   ✓ Implement retention policy

6. VALIDATION
   ✓ Always verify loaded data
   ✓ Check data integrity checksums
   ✓ Compare metrics pre/post export
   ✓ Test in staging before production

7. DOCUMENTATION
   ✓ Document export date/time
   ✓ Note Python/package versions
   ✓ Include data descriptions
   ✓ Explain any preprocessing
   ✓ List assumptions and constraints
"""

print(best_practices)


BEST PRACTICES FOR EXPORT & SERIALIZATION:

1. ORGANIZATION
   ✓ Use consistent directory structure
   ✓ Include metadata (dates, versions)
   ✓ Document format choices
   ✓ Version all artifacts

2. CSV EXPORTS
   ✓ Always export cuts.csv first
   ✓ Then segmented_data.csv for verification
   ✓ Include segment_metrics.csv for documentation
   ✓ Test import immediately after export

3. JSON EXPORTS
   ✓ Use optimizer.to_json() for portability
   ✓ Separately export metrics.json
   ✓ Document schema and versions
   ✓ Validate with JSON schema

4. PICKLE EXPORTS
   ✓ Use only in trusted environments
   ✓ Document Python version used
   ✓ Test loading on target environment
   ✓ Consider security implications

5. FILE MANAGEMENT
   ✓ Use consistent naming: {timestamp}_{modeltype}.pkl
   ✓ Compress older exports
   ✓ Archive with metadata
   ✓ Implement retention policy

6. VALIDATION
   ✓ Always verify loaded data
   ✓ Check data integrity checksums
   ✓ Compare metrics pre/post export
  

## Key Takeaways

✅ **Export Formats:**
- CSV: For sharing and analysis
- JSON: For portability and integration
- Pickle: For Python-only workflows

✅ **Export Functions:**
- `export_segmentation_to_csv()` - Multi-file CSV export
- `optimizer.to_json()` - JSON serialization
- `save_optimizer_state()` - Pickle export

✅ **Import Functions:**
- `import_segmentation_from_csv()` - Load from CSV
- `SegmentationOptimizer.from_json()` - JSON deserialization
- `load_optimizer_state()` - Pickle import

✅ **Validation:**
- Always verify after loading
- Check data integrity
- Compare pre/post export metrics